# A2 — Knowledge-Base Demo (fill this)
Show OCR quality on a sample and one working retrieval example.

# A2 — Knowledge-Base Demo

This notebook demonstrates the constructed knowledge base through:

1. **OCR quality** on a small sample of indexed chunks.
2. **Index statistics**:
   - number of chunks indexed
   - embedding dimension
   - FAISS index type
   - corpus coverage: pages and words indexed vs. total
3. **One working retrieval example**:
   - a real user query
   - top retrieved chunk
   - source book/page
   - whether the retrieved page contains the relevant answer

In [1]:
# IMPLEMENT: run OCR quality + one retrieval, end to end
from pathlib import Path
import json
import re
import random

import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from IPython.display import display, Markdown

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

INDEX_DIR = Path("../data/interim/index")

FAISS_PATH = INDEX_DIR / "index.faiss"
CHUNKS_PATH = INDEX_DIR / "chunks.jsonl"

assert FAISS_PATH.exists(), f"Missing FAISS index: {FAISS_PATH}"
assert CHUNKS_PATH.exists(), f"Missing chunk metadata: {CHUNKS_PATH}"

print(f"Index:  {FAISS_PATH}")
print(f"Chunks: {CHUNKS_PATH}")

/Users/mesbah/Desktop/Agri-Bot/CSE429-Project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Index:  ../data/interim/index/index.faiss
Chunks: ../data/interim/index/chunks.jsonl


In [2]:
# Load FAISS index
index = faiss.read_index(str(FAISS_PATH))

# Load chunk metadata
chunks = []

with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))

print(f"Loaded {len(chunks):,} chunk records")
print(f"FAISS contains {index.ntotal:,} vectors")

assert index.ntotal == len(chunks), (
    f"Mismatch: FAISS has {index.ntotal} vectors but "
    f"chunks.jsonl has {len(chunks)} records"
)

Loaded 1,814 chunk records
FAISS contains 1,814 vectors


In [3]:
# ------------------------------------------------------------
# Basic index statistics
# ------------------------------------------------------------

n_chunks = index.ntotal
embedding_dim = index.d

index_type = type(index).__name__

print("=== Index Statistics ===")
print(f"Chunks indexed       : {n_chunks:,}")
print(f"Embedding dimension  : {embedding_dim}")
print(f"Index type            : {index_type}")
print(f"Index metric          : Inner Product")

=== Index Statistics ===
Chunks indexed       : 1,814
Embedding dimension  : 1024
Index type            : IndexFlatIP
Index metric          : Inner Product


In [4]:
# ------------------------------------------------------------
# Pages represented in the index
# ------------------------------------------------------------

indexed_page_ids = set()

for chunk in chunks:
    indexed_page_ids.update(chunk.get("page_ids", []))

indexed_pages = len(indexed_page_ids)


# ------------------------------------------------------------
# Words represented in the index
# ------------------------------------------------------------

def count_words(text):
    """
    Unicode-friendly approximate word count.

    For Bengali/English OCR text, whitespace-separated tokens are
    a useful corpus-level measure.
    """
    return len(text.split())


indexed_words = sum(
    count_words(chunk.get("text", ""))
    for chunk in chunks
)

print(f"Pages represented in index : {indexed_pages:,}")
print(f"Words represented in index : {indexed_words:,}")

Pages represented in index : 742
Words represented in index : 129,496


In [5]:
# ------------------------------------------------------------
# Total corpus from OCR region caches
# ------------------------------------------------------------

OCR_DIR = Path("../data/interim/ocr")

assert OCR_DIR.exists(), f"OCR directory not found: {OCR_DIR}"


def load_ocr_pages(ocr_dir):
    """
    Return:
        total_pages
        total_words
        page_ids
    based on OCR region caches.
    """

    page_ids = set()
    total_words = 0

    json_files = sorted(ocr_dir.glob("*/*.json"))

    for path in json_files:
        doc_id = path.parent.name
        page_stem = path.stem

        page_id = f"{doc_id}:{page_stem}"
        page_ids.add(page_id)

        try:
            with open(path, "r", encoding="utf-8") as f:
                regions = json.load(f)
        except Exception:
            continue

        for region in regions:
            text = region.get("corrected_text", "")
            total_words += count_words(text)

    return len(page_ids), total_words, page_ids


total_pages, total_words, all_page_ids = load_ocr_pages(OCR_DIR)

print(f"Total OCR pages  : {total_pages:,}")
print(f"Total OCR words  : {total_words:,}")

Total OCR pages  : 742
Total OCR words  : 131,123


In [6]:
page_coverage = (
    100 * indexed_pages / total_pages
    if total_pages else 0
)

word_coverage = (
    100 * indexed_words / total_words
    if total_words else 0
)

coverage_df = pd.DataFrame([
    {
        "Metric": "Pages",
        "Indexed": indexed_pages,
        "Total": total_pages,
        "Coverage (%)": page_coverage,
    },
    {
        "Metric": "Words",
        "Indexed": indexed_words,
        "Total": total_words,
        "Coverage (%)": word_coverage,
    },
])

coverage_df["Coverage (%)"] = coverage_df["Coverage (%)"].round(2)

display(coverage_df)

,Metric,Indexed,Total,Coverage (%)
0,Pages,742,742,100.00
1,Words,129496,131123,98.76


In [7]:
# ------------------------------------------------------------
# OCR quality sample
# ------------------------------------------------------------

random.seed(42)

sample_n = min(5, len(chunks))
sample_chunks = random.sample(chunks, sample_n)

for i, chunk in enumerate(sample_chunks, 1):

    page = chunk["page_ids"][0] if chunk.get("page_ids") else "unknown"

    print("=" * 90)
    print(f"Sample {i}")
    print(f"Chunk ID : {chunk['id']}")
    print(f"Page     : {page}")
    print("-" * 90)
    print(chunk["text"])
    print()

Sample 1
Chunk ID : Krishi-Bigyan:page-272:1-0
Page     : Krishi-Bigyan:page-272
------------------------------------------------------------------------------------------
ইচ্ছার সহিত জড়িত এবং উহা ঐ সকল ইজ্জার প্রভাবের তারতম্য-
অনুসারে বিভিন্ন রূপ হইয়া থাকে। কোন দ্রব্যের মূল্য বাজার
অপেক্ষাও অধিক হইতে পারে; কিন্তু এই প্রকার মূল্যের আধিক্য
ক্রেতার ইচ্ছার বলবশ্যীর উপর নির্ভর করে; যথা, যখন চাউলের
দর টাকায় = ৮ সের, এক ব্যক্তির তখন নিজ পারিবারিক খাদ্যের
জন্য দৈনিক = ৪ সের চাউলের প্রয়োজন। যদি চাউল মহার্য হইয়া
টাকায় = ১ সেরে পরিণত হয়, তাহা হইলে হয় তাহাকে ঐ = ৪ সের
চাউলের জন্য পূর্ব্বাপেক্ষা অধিক ব্যয় করিতে হইবে, অথবা তাহাকে
১৪ সের অপেক্ষা কম চাউল ক্রয় করিতে হইবে। এ ক্ষেত্রে ইচ্ছার
বিরুদ্ধে কার্য্য করিতে হইলেও এই বিরুদ্ধতার মীমাংসা আপোষেই
হইয়া থাকে। তাহাকে চাউলও অল্প ক্রয় করিতে হয়, অথচ অর্থও
পূর্ব্বাপেক্ষা অধিক ব্যয় করিতে হয়। বাজার-দর বুদ্ধি পাইলে
সাধারণতঃ চাহিদার হ্রাস হয়। মূল্যবৃদ্ধির কারণ ইহাতে বুঝা যায়
না। ইহা হইতে প্রতীয়মান হয় যে মূল্যের হবাস-বৃদ্ধির সহিত চাদার
পরিমাণের

In [19]:
print(f"Showing {min(4, len(sample_chunks))} OCR examples:\n")

for i, c in enumerate(sample_chunks[:4], start=1):
    print("=" * 100)
    print(f"Example {i}")
    print(f"Chunk ID : {c['id']}")
    print(f"Page     : {c['page_ids'][0] if c.get('page_ids') else ''}")
    print("-" * 100)
    print(c["text"])
    print()

Showing 4 OCR examples:

Example 1
Chunk ID : Krishi-Bigyan:page-272:1-0
Page     : Krishi-Bigyan:page-272
----------------------------------------------------------------------------------------------------
ইচ্ছার সহিত জড়িত এবং উহা ঐ সকল ইজ্জার প্রভাবের তারতম্য-
অনুসারে বিভিন্ন রূপ হইয়া থাকে। কোন দ্রব্যের মূল্য বাজার
অপেক্ষাও অধিক হইতে পারে; কিন্তু এই প্রকার মূল্যের আধিক্য
ক্রেতার ইচ্ছার বলবশ্যীর উপর নির্ভর করে; যথা, যখন চাউলের
দর টাকায় = ৮ সের, এক ব্যক্তির তখন নিজ পারিবারিক খাদ্যের
জন্য দৈনিক = ৪ সের চাউলের প্রয়োজন। যদি চাউল মহার্য হইয়া
টাকায় = ১ সেরে পরিণত হয়, তাহা হইলে হয় তাহাকে ঐ = ৪ সের
চাউলের জন্য পূর্ব্বাপেক্ষা অধিক ব্যয় করিতে হইবে, অথবা তাহাকে
১৪ সের অপেক্ষা কম চাউল ক্রয় করিতে হইবে। এ ক্ষেত্রে ইচ্ছার
বিরুদ্ধে কার্য্য করিতে হইলেও এই বিরুদ্ধতার মীমাংসা আপোষেই
হইয়া থাকে। তাহাকে চাউলও অল্প ক্রয় করিতে হয়, অথচ অর্থও
পূর্ব্বাপেক্ষা অধিক ব্যয় করিতে হয়। বাজার-দর বুদ্ধি পাইলে
সাধারণতঃ চাহিদার হ্রাস হয়। মূল্যবৃদ্ধির কারণ ইহাতে বুঝা যায়
না। ইহা হইতে প্রতীয়মান হয় যে মূল্

In [9]:
# ------------------------------------------------------------
# Embedding model used by the KB
# ------------------------------------------------------------

MODEL_NAME = "BAAI/bge-m3"

embedder = SentenceTransformer(MODEL_NAME)

print(f"Loaded embedding model: {MODEL_NAME}")
print(f"Embedding dimension: {embedder.get_sentence_embedding_dimension()}")

assert (
    embedder.get_sentence_embedding_dimension()
    == index.d
), "Embedding dimension does not match FAISS index!"

Loaded embedding model: BAAI/bge-m3
Embedding dimension: 1024


In [10]:
def retrieve(query: str, k: int = 5):
    """
    Retrieve top-k chunks using the same embedding configuration
    used during index construction.
    """

    query_vector = embedder.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)

    scores, indices = index.search(query_vector, k)

    results = []

    for rank, (score, idx) in enumerate(
        zip(scores[0], indices[0]),
        start=1
    ):
        if idx < 0:
            continue

        chunk = chunks[int(idx)].copy()

        results.append({
            "rank": rank,
            "index_row": int(idx),
            "score": float(score),
            "chunk": chunk,
        })

    return results

In [14]:
# ------------------------------------------------------------
# One verified working retrieval example
# ------------------------------------------------------------

QUERY = "কোন কোন বৈশিষ্ট্যের দ্বারা মৃত্তিকার উর্বরতা ও অনুর্বরতা নির্দেশ করা যায় এবং কোন মৃত্তিকা কোন ফসলের জন্য উপযোগী?"

results = retrieve(QUERY, k=5)

print(f"Query: {QUERY}")
print()

Query: কোন কোন বৈশিষ্ট্যের দ্বারা মৃত্তিকার উর্বরতা ও অনুর্বরতা নির্দেশ করা যায় এবং কোন মৃত্তিকা কোন ফসলের জন্য উপযোগী?



In [15]:
top = results[0]
top_chunk = top["chunk"]

print("=" * 90)
print("TOP RETRIEVED CHUNK")
print("=" * 90)

print(f"Rank       : {top['rank']}")
print(f"FAISS row  : {top['index_row']}")
print(f"Score      : {top['score']:.4f}")
print(f"Chunk ID   : {top_chunk['id']}")
print(f"Book       : {top_chunk['doc_id']}")
print(f"Page       : {top_chunk['page_ids']}")
print()
print(top_chunk["text"])

TOP RETRIEVED CHUNK
Rank       : 1
FAISS row  : 1120
Score      : 0.7303
Chunk ID   : Krishi-Bigyan:page-206:1-0
Book       : Krishi-Bigyan
Page       : ['Krishi-Bigyan:page-206']

মৃত্তিকার বর্ণ এবং যাত্ররক অবস্থার প্রতি লক্ষ্য করিয়াও উহার
উর্বরতা এবং অনুর্ব্বরা নির্দেশ করা যায়। কালো এবঙ গীত বর্ণের
মৃত্তিকা সাধারণতঃ উর্ব্বরা হইয়া থাকে; এবং সাদা, ধূমর ও অধিক লাল
বর্ণের মৃত্তিকা সাধারণতঃ অঙ্কুর হইয়া থাকে। যে মৃৎকার শীতকালে,
অর্থাৎ নিতান্ত শুষ্ক দিনেও, লাঙ্গল দ্বারা অনায়াসে কর্ষণ করা যায় এইরূপ
হাল্কা মৃত্তিকা স্বভাবতঃই উর্বরা। বৃষ্টিপাত না হইলে যে মৃৎকার
সহজে কর্ষণ করা যায় না এইরূপ দৃঢ় মৃত্তিকা অধিকাংশ স্থলেই উর্বরা হয়
না। বৃষ্টিবারি-পতন মাত্রই যে জমি হইতে নিঃসৃত হইয়া যায় এবং যে
জমিতে বৃষ্টিবারি অধিককাল দাঁড়াইয়া থাকে, এই উভয় প্রকার মৃত্তিকাই
উর্বর হইলেও কৃষিকার্য্যের পক্ষে উপযোগী নহে। যে মৃত্তিকা বৃষ্টিবারি
দ্বারা বিগলিত ও বিধৌত হইয়া স্থানান্তরে চলিয়া যায় তাহাও কৃষিকার্য্যের
উপযুক্ত নহে। কঠিন এবং হাল্কা—এই উভয় প্রকার মৃত্তিকাতে আপন
আপন স্বভাবের উপযোগী ফসল জন্মিতে পারে; 

In [20]:
print(f"Showing {min(4, len(results))} retrieved chunks:\n")

for r in results[:4]:
    chunk = r["chunk"]

    print("=" * 100)
    print(f"Rank  : {r['rank']}")
    print(f"Score : {r['score']:.4f}")
    print(f"Book  : {chunk['doc_id']}")
    print(f"Page  : {', '.join(chunk.get('page_ids', []))}")
    print("-" * 100)
    print(chunk["text"])
    print()

Showing 4 retrieved chunks:

Rank  : 1
Score : 0.7303
Book  : Krishi-Bigyan
Page  : Krishi-Bigyan:page-206
----------------------------------------------------------------------------------------------------
মৃত্তিকার বর্ণ এবং যাত্ররক অবস্থার প্রতি লক্ষ্য করিয়াও উহার
উর্বরতা এবং অনুর্ব্বরা নির্দেশ করা যায়। কালো এবঙ গীত বর্ণের
মৃত্তিকা সাধারণতঃ উর্ব্বরা হইয়া থাকে; এবং সাদা, ধূমর ও অধিক লাল
বর্ণের মৃত্তিকা সাধারণতঃ অঙ্কুর হইয়া থাকে। যে মৃৎকার শীতকালে,
অর্থাৎ নিতান্ত শুষ্ক দিনেও, লাঙ্গল দ্বারা অনায়াসে কর্ষণ করা যায় এইরূপ
হাল্কা মৃত্তিকা স্বভাবতঃই উর্বরা। বৃষ্টিপাত না হইলে যে মৃৎকার
সহজে কর্ষণ করা যায় না এইরূপ দৃঢ় মৃত্তিকা অধিকাংশ স্থলেই উর্বরা হয়
না। বৃষ্টিবারি-পতন মাত্রই যে জমি হইতে নিঃসৃত হইয়া যায় এবং যে
জমিতে বৃষ্টিবারি অধিককাল দাঁড়াইয়া থাকে, এই উভয় প্রকার মৃত্তিকাই
উর্বর হইলেও কৃষিকার্য্যের পক্ষে উপযোগী নহে। যে মৃত্তিকা বৃষ্টিবারি
দ্বারা বিগলিত ও বিধৌত হইয়া স্থানান্তরে চলিয়া যায় তাহাও কৃষিকার্য্যের
উপযুক্ত নহে। কঠিন এবং হাল্কা—এই উভয় প্রকার মৃত্তিকাতে আপন
আপন স্বভাবে